# Gold Layer — Metadata Compliance Engine

**Catalog:** metadata_governance  
**Schema:** gold  
**Tables:** gold_metadata, gold_certified_metadata, gold_quality_metrics  
**Layer:** Gold (certified, governed, compliance-scored metadata)  
**Source:** metadata_governance.silver.silver_metadata + metadata_governance.silver.bronze_metadata_quarantine  
**Governance Rules:** R01–R12 (Completeness, Ownership, Security, Structure, Privacy, Business Glossary)  
**Purpose:** Promote clean Silver metadata to Gold, apply 9 governance rules, assign PASS/FAIL certification status, calculate quality scores, and write certified results to Gold layer for Power BI dashboards and Genie AI.


## Step 0: Promote Silver to Gold Base Table

Write clean Silver metadata to the Gold layer as a base table. This matches the architecture where the Compliance Engine reads from Gold — not directly from Silver.

In [0]:
from pyspark.sql.functions import (
    col, when, lit, concat_ws, length, trim,
    avg, count, sum as spark_sum, current_timestamp,
    lower, regexp_extract
)

silver_df = spark.table("metadata_governance.silver.silver_metadata")
silver_count = silver_df.count()

spark.sql("DROP TABLE IF EXISTS metadata_governance.gold.gold_metadata")

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("metadata_governance.gold.gold_metadata")

gold_check = spark.table("metadata_governance.gold.gold_metadata").count()

assert silver_count == gold_check, \
    f"Row count mismatch! Silver={silver_count} Gold={gold_check}"
print(f"  Row count assertion PASSED — {gold_check} rows")

print("=" * 60)
print("  STEP 0 — Silver Promoted to Gold")
print("=" * 60)
print(f"  Silver rows read      : {silver_count}")
print(f"  Gold rows written     : {gold_check}")
print(f"  Table                 : metadata_governance.gold.gold_metadata ")
print("=" * 60)

## Step 1: Read Metadata from Gold Layer

Read the promoted metadata from the Gold layer. This is the official input to the Compliance Engine, matching the architecture diagram where the Compliance Engine sits to the right of Gold.

In [0]:
gold_df = spark.table("metadata_governance.gold.gold_metadata")
gold_count = gold_df.count()

print("=" * 60)
print("  STEP 1 — Gold Layer Loaded")
print("=" * 60)
print(f"  Gold rows loaded      : {gold_count}")
print(f"  Columns available     : {len(gold_df.columns)}")
print("=" * 60)

display(gold_df.limit(5))

## Step 2: Read Failed Records from Quarantine

Pipeline sent 457 invalid records to the quarantine table via the Alert path. The Compliance Engine must also process these records and flag them with their ingestion failure reason plus governance verdict on top.

In [0]:
quarantine_df = spark.table("metadata_governance.silver.bronze_metadata_quarantine")
quarantine_count = quarantine_df.count()

expected_total = 10000
actual_total = gold_count + quarantine_count
assert actual_total == expected_total, \
    f"Total count mismatch! Expected {expected_total} got {actual_total}"
print(f"  Total count assertion PASSED - {actual_total} records")

print("=" * 60)
print("  STEP 2 - Quarantine Records Loaded")
print("=" * 60)
print(f"  Gold rows             : {gold_count}")
print(f"  Quarantine rows       : {quarantine_count}")
print(f"  Total to process      : {actual_total}")
print("=" * 60)

display(quarantine_df.limit(5))

## Step 3: Define Governance Rules R01-R12

Define the 12 governance rules every metadata record must satisfy. These rules are applied to both Gold records and Quarantine records.


In [0]:
valid_security  = ["public", "internal", "confidential", "restricted"]
valid_schemas   = ["bronze", "silver", "gold"]
pii_restricted  = ["confidential", "restricted"]

required_cols = [
    "column_name", "column_desc", "table_desc", "data_steward",
    "security_classification", "schema_name", "total_record_count",
    "pii_flag", "critical_data_element_flag", "term_name",
    "table_owner_in_source", "tag_value"
]
missing = [c for c in required_cols if c not in gold_df.columns]
assert len(missing) == 0, f"Missing required columns: {missing}"
print(f"  Schema validation PASSED — All {len(required_cols)} required columns present")

# R01 — Completeness
r01_col_desc   = col("column_desc").isNotNull() & (length(trim(col("column_desc"))) >= 10)
r01_table_desc = col("table_desc").isNotNull() & (length(trim(col("table_desc"))) >= 10)

# R02 — Ownership
r02_steward = col("data_steward").isNotNull() & (trim(col("data_steward")) != "")

# R03 — Security
r03_security = col("security_classification").isin(valid_security)

# R04 — Structure (hard override)
r04_schema = col("schema_name").isin(valid_schemas)

# R05 — Structure
r05_row_count = col("total_record_count").isNotNull() & (col("total_record_count") > 0)

# R06 — Privacy
r06_pii = col("pii_flag").isNotNull()

# R07 — Business
r07_cde = col("critical_data_element_flag").isNotNull()

# R08 — Business Glossary
r08_term = col("term_name").isNotNull() & (trim(col("term_name")) != "")

# R09 — Cross-field Privacy
r09_pii_security = (
    (col("pii_flag") != True) |
    (col("security_classification").isin(pii_restricted))
)

# R10 — Naming Convention
r10_naming = col("column_name").rlike("^[a-z][a-z0-9_]*$")

# R11 — Source Ownership
r11_table_owner = col("table_owner_in_source").isNotNull() & \
                  (trim(col("table_owner_in_source")) != "")

# R12 — Tagging
r12_tag = col("tag_value").isNotNull() & (trim(col("tag_value")) != "")

print("=" * 60)
print("  STEP 3 — Governance Rules Defined (R01-R12)")
print("=" * 60)
print(f"  R01  → column_desc + table_desc: min 10 chars      [Completeness]")
print(f"  R02  → data_steward: not null or blank             [Ownership]")
print(f"  R03  → security_classification: approved values    [Security]")
print(f"  R04  → schema_name: bronze/silver/gold (override)  [Structure]")
print(f"  R05  → total_record_count: not null and > 0        [Structure]")
print(f"  R06  → pii_flag: not null                          [Privacy]")
print(f"  R07  → critical_data_element_flag: not null        [Business]")
print(f"  R08  → term_name: not null or blank                [Glossary]")
print(f"  R09  → pii=true → confidential/restricted          [Cross-field]")
print(f"  R10  → column_name: lowercase + underscores only   [Structure]")
print(f"  R11  → table_owner_in_source: not null             [Ownership]")
print(f"  R12  → tag_value: not null or blank                [Tagging]")
print("=" * 60)

## Step 4: Apply Governance Rules to Gold Records

Apply R01–R12 to every Gold record. Assign certification_status, error_detail, remediation_hint, completeness_score and ingestion_status.


In [0]:
def apply_governance_rules(df, ingestion_status_value):
    """
    Apply R01-R12 governance rules to a metadata DataFrame.

    Args:
        df: Input PySpark DataFrame with metadata records
        ingestion_status_value: PASSED_SILVER or FAILED_SILVER

    Returns:
        DataFrame with 5 new governance columns:
        error_detail, remediation_hint, certification_status,
        completeness_score, ingestion_status
    """
    return df \
        .withColumn(
            "error_detail",
            concat_ws(" | ",
                when(~r01_col_desc,     lit("R01: column_desc missing or too short")),
                when(~r01_table_desc,   lit("R01: table_desc missing or too short")),
                when(~r02_steward,      lit("R02: data_steward missing")),
                when(~r03_security,     lit("R03: security_classification missing or invalid")),
                when(~r04_schema,       lit("R04: schema_name not recognised")),
                when(~r05_row_count,    lit("R05: table has no rows or row count is null")),
                when(~r06_pii,          lit("R06: pii_flag is not set")),
                when(~r07_cde,          lit("R07: critical_data_element_flag is not set")),
                when(~r08_term,         lit("R08: no business term assigned")),
                when(~r09_pii_security, lit("R09: PII column must have confidential or restricted classification")),
                when(~r10_naming,       lit("R10: column_name does not follow naming convention")),
                when(~r11_table_owner,  lit("R11: table_owner_in_source is missing")),
                when(~r12_tag,          lit("R12: tag_value is not set"))
            )
        ) \
        .withColumn(
            "remediation_hint",
            concat_ws(" | ",
                when(~r01_col_desc,     lit("Add a column description of at least 10 characters")),
                when(~r01_table_desc,   lit("Add a table description of at least 10 characters")),
                when(~r02_steward,      lit("Assign a responsible data steward or team")),
                when(~r03_security,     lit("Set security_classification to: public, internal, confidential or restricted")),
                when(~r04_schema,       lit("Move table to correct schema: bronze, silver or gold")),
                when(~r05_row_count,    lit("Investigate this table — it appears to be empty")),
                when(~r06_pii,          lit("Set pii_flag to true or false for this column")),
                when(~r07_cde,          lit("Set critical_data_element_flag to true or false")),
                when(~r08_term,         lit("Assign a business glossary term to this column")),
                when(~r09_pii_security, lit("PII data must be classified as confidential or restricted")),
                when(~r10_naming,       lit("Rename column to use only lowercase letters, numbers and underscores")),
                when(~r11_table_owner,  lit("Assign a source system owner to this table")),
                when(~r12_tag,          lit("Assign a domain tag value to this column"))
            )
        ) \
        .withColumn(
            "certification_status",
            when(
                r01_col_desc & r01_table_desc & r02_steward & r03_security &
                r04_schema & r05_row_count & r06_pii & r07_cde & r08_term &
                r09_pii_security & r10_naming & r11_table_owner & r12_tag,
                lit("PASS")
            ).otherwise(lit("FAIL"))
        ) \
        .withColumn("r01_col_pass",    when(r01_col_desc,     lit(1)).otherwise(lit(0))) \
        .withColumn("r01_table_pass",  when(r01_table_desc,   lit(1)).otherwise(lit(0))) \
        .withColumn("r02_pass",        when(r02_steward,      lit(1)).otherwise(lit(0))) \
        .withColumn("r03_pass",        when(r03_security,     lit(1)).otherwise(lit(0))) \
        .withColumn("r05_pass",        when(r05_row_count,    lit(1)).otherwise(lit(0))) \
        .withColumn("r06_pass",        when(r06_pii,          lit(1)).otherwise(lit(0))) \
        .withColumn("r07_pass",        when(r07_cde,          lit(1)).otherwise(lit(0))) \
        .withColumn("r08_pass",        when(r08_term,         lit(1)).otherwise(lit(0))) \
        .withColumn("r09_pass",        when(r09_pii_security, lit(1)).otherwise(lit(0))) \
        .withColumn("r10_pass",        when(r10_naming,       lit(1)).otherwise(lit(0))) \
        .withColumn("r11_pass",        when(r11_table_owner,  lit(1)).otherwise(lit(0))) \
        .withColumn("r12_pass",        when(r12_tag,          lit(1)).otherwise(lit(0))) \
        .withColumn(
            "completeness_score",
            (
                col("r01_col_pass")  + col("r01_table_pass") +
                col("r02_pass")      + col("r03_pass") +
                col("r05_pass")      + col("r06_pass") +
                col("r07_pass")      + col("r08_pass") +
                col("r09_pass")      + col("r10_pass") +
                col("r11_pass")      + col("r12_pass")
            ) * lit(100) / lit(12)
        ) \
        .withColumn("completeness_score", col("completeness_score").cast("string")) \
        .drop(
            "r01_col_pass", "r01_table_pass", "r02_pass", "r03_pass",
            "r05_pass", "r06_pass", "r07_pass", "r08_pass", "r09_pass",
            "r10_pass", "r11_pass", "r12_pass"
        ) \
        .withColumn(
            "error_detail",
            when(col("certification_status") == "PASS", lit(None)).otherwise(col("error_detail"))
        ) \
        .withColumn(
            "remediation_hint",
            when(col("certification_status") == "PASS", lit(None)).otherwise(col("remediation_hint"))
        ) \
        .withColumn("ingestion_status", lit(ingestion_status_value))

df_gold_certified = apply_governance_rules(gold_df, "PASSED_SILVER")

pass_gold = df_gold_certified.filter(col("certification_status") == "PASS").count()
fail_gold = df_gold_certified.filter(col("certification_status") == "FAIL").count()
pass_rate = int(pass_gold) / int(gold_count) * 100

print("=" * 60)
print("  STEP 4 — Gold Records Certification Results")
print("=" * 60)
print(f"  Total records         : {gold_count}")
print(f"  PASS                  : {pass_gold}")
print(f"  FAIL                  : {fail_gold}")
print(f"  Pass rate             : {pass_rate:.2f}%")
print("=" * 60)

df_gold_certified.select(
    "table_name", "column_name",
    "certification_status", "error_detail", "ingestion_status"
).show(10, truncate=False)

## Step 5: Apply Governance Rules to Quarantine Records

Apply R01-R12 to failed records. These records automatically receive FAIL certification and are flagged with both ingestion failure reason and any additional governance failures.

In [0]:
df_quarantine_certified = apply_governance_rules(quarantine_df, "FAILED_SILVER")

df_quarantine_certified = df_quarantine_certified.withColumn(
    "certification_status", lit("FAIL")
).withColumn(
    "error_detail",
    concat_ws(" | ",
        concat_ws(": ", lit("INGESTION FAIL"), col("failure_reason")),
        col("error_detail")
    )
)

df_quarantine_final = df_quarantine_certified.drop("failure_reason")

quarantine_pass_check = df_quarantine_final.filter(
    col("certification_status") == "PASS"
).count()
assert quarantine_pass_check == 0, \
    f"Quarantine records incorrectly marked as PASS: {quarantine_pass_check}"
print(f"  Quarantine FAIL assertion PASSED  — All {quarantine_count} records are FAIL")

print("=" * 60)
print("  STEP 5 — Quarantine Records Certification Results")
print("=" * 60)
print(f"  Total quarantine records  : {quarantine_count}")
print(f"  All records               : FAIL (failed Silver ingestion)")
print("=" * 60)

df_quarantine_final.select(
    "table_name", "column_name",
    "certification_status", "error_detail", "ingestion_status"
).show(10, truncate=False)

## Step 6: Combine Gold + Quarantine into Complete Dataset

Union Gold certified records and Quarantine certified records into one complete dataset covering all 10,000 original records.


In [0]:
df_full = df_gold_certified.unionByName(
    df_quarantine_final,
    allowMissingColumns=True
)

total_full = df_full.count()
pass_full  = df_full.filter(col("certification_status") == "PASS").count()
fail_full  = df_full.filter(col("certification_status") == "FAIL").count()
overall_rate = int(pass_full) / int(total_full) * 100

assert total_full == gold_count + quarantine_count, \
    f"Union row count mismatch! Expected {gold_count + quarantine_count} got {total_full}"
print(f"  Union row count assertion PASSED — {total_full} total records")

assert pass_full + fail_full == total_full, \
    "PASS + FAIL counts do not equal total records!"
print(f"  PASS + FAIL assertion PASSED — {pass_full} + {fail_full} = {total_full}")

print("=" * 60)
print("  STEP 6 — COMPLETE COMPLIANCE PICTURE")
print("=" * 60)
print(f"  Gold PASS                 : {pass_gold}")
print(f"  Gold FAIL                 : {fail_gold}")
print(f"  Quarantine FAIL           : {quarantine_count}")
print(f"  Total records processed   : {total_full}")
print(f"  Overall PASS              : {pass_full}")
print(f"  Overall FAIL              : {fail_full}")
print(f"  Overall compliance score  : {overall_rate:.2f}%")
print("=" * 60)

# Step 7: Calculate Quality Metrics

Calculate completeness scores at 4 levels: field level, table level, schema level, database level, and overall catalog compliance score.

In [0]:
def safe_num(value, default=0.0):
    return float(value) if value is not None else default

def status_label(score):
    if score >= 80:
        return "GOOD"
    elif score >= 50:
        return "NEEDS REVIEW"
    else:
        return "CRITICAL"


# Field level metrics
field_metrics = df_full.agg(
    avg(when(r01_col_desc,     lit(100)).otherwise(lit(0))).alias("R01_column_desc_pct"),
    avg(when(r01_table_desc,   lit(100)).otherwise(lit(0))).alias("R01_table_desc_pct"),
    avg(when(r02_steward,      lit(100)).otherwise(lit(0))).alias("R02_data_steward_pct"),
    avg(when(r03_security,     lit(100)).otherwise(lit(0))).alias("R03_security_classification_pct"),
    avg(when(r04_schema,       lit(100)).otherwise(lit(0))).alias("R04_schema_name_pct"),
    avg(when(r05_row_count,    lit(100)).otherwise(lit(0))).alias("R05_row_count_pct"),
    avg(when(r06_pii,          lit(100)).otherwise(lit(0))).alias("R06_pii_flag_pct"),
    avg(when(r07_cde,          lit(100)).otherwise(lit(0))).alias("R07_critical_flag_pct"),
    avg(when(r08_term,         lit(100)).otherwise(lit(0))).alias("R08_term_name_pct"),
    avg(when(r09_pii_security, lit(100)).otherwise(lit(0))).alias("R09_pii_security_pct"),
    avg(when(r10_naming,       lit(100)).otherwise(lit(0))).alias("R10_naming_convention_pct"),
    avg(when(r11_table_owner,  lit(100)).otherwise(lit(0))).alias("R11_table_owner_pct"),
    avg(when(r12_tag,          lit(100)).otherwise(lit(0))).alias("R12_tag_value_pct")
)

field_row = field_metrics.collect()[0]

fields = [
    ("R01 column description",        safe_num(field_row["R01_column_desc_pct"])),
    ("R01 table description",         safe_num(field_row["R01_table_desc_pct"])),
    ("R02 data steward",              safe_num(field_row["R02_data_steward_pct"])),
    ("R03 security classification",   safe_num(field_row["R03_security_classification_pct"])),
    ("R04 schema name",               safe_num(field_row["R04_schema_name_pct"])),
    ("R05 row count",                 safe_num(field_row["R05_row_count_pct"])),
    ("R06 PII flag",                  safe_num(field_row["R06_pii_flag_pct"])),
    ("R07 critical data flag",         safe_num(field_row["R07_critical_flag_pct"])),
    ("R08 business term name",         safe_num(field_row["R08_term_name_pct"])),
    ("R09 PII security alignment",     safe_num(field_row["R09_pii_security_pct"])),
    ("R10 naming convention",          safe_num(field_row["R10_naming_convention_pct"])),
    ("R11 table owner",                safe_num(field_row["R11_table_owner_pct"])),
    ("R12 tag value",                  safe_num(field_row["R12_tag_value_pct"])),
]

worst_field = min(fields, key=lambda x: x[1])
best_field = max(fields, key=lambda x: x[1])

print("=" * 70)
print("FIELD LEVEL METRICS — ALL GOVERNANCE RULES")
print("=" * 70)

for name, score in fields:
    print(f"{status_label(score):<14} | {name:<35} | {score:>6.2f}%")

print("=" * 70)
print(f"Best performing field  : {best_field[0]} at {best_field[1]:.2f}%")
print(f"Biggest governance gap : {worst_field[0]} at {worst_field[1]:.2f}%")
print("=" * 70)


# Rule breakdown
rule_breakdown = df_full.agg(
    spark_sum(when(~r01_col_desc,     lit(1)).otherwise(lit(0))).alias("R01_col_desc_failures"),
    spark_sum(when(~r01_table_desc,   lit(1)).otherwise(lit(0))).alias("R01_table_desc_failures"),
    spark_sum(when(~r02_steward,      lit(1)).otherwise(lit(0))).alias("R02_steward_failures"),
    spark_sum(when(~r03_security,     lit(1)).otherwise(lit(0))).alias("R03_security_failures"),
    spark_sum(when(~r04_schema,       lit(1)).otherwise(lit(0))).alias("R04_schema_failures"),
    spark_sum(when(~r05_row_count,    lit(1)).otherwise(lit(0))).alias("R05_row_count_failures"),
    spark_sum(when(~r06_pii,          lit(1)).otherwise(lit(0))).alias("R06_pii_failures"),
    spark_sum(when(~r07_cde,          lit(1)).otherwise(lit(0))).alias("R07_cde_failures"),
    spark_sum(when(~r08_term,         lit(1)).otherwise(lit(0))).alias("R08_term_failures"),
    spark_sum(when(~r09_pii_security, lit(1)).otherwise(lit(0))).alias("R09_pii_security_failures"),
    spark_sum(when(~r10_naming,       lit(1)).otherwise(lit(0))).alias("R10_naming_failures"),
    spark_sum(when(~r11_table_owner,  lit(1)).otherwise(lit(0))).alias("R11_table_owner_failures"),
    spark_sum(when(~r12_tag,          lit(1)).otherwise(lit(0))).alias("R12_tag_failures")
)

bd_row = rule_breakdown.collect()[0]

rule_failures = [
    ("R01 column description",      int(safe_num(bd_row["R01_col_desc_failures"]))),
    ("R01 table description",       int(safe_num(bd_row["R01_table_desc_failures"]))),
    ("R02 data steward",            int(safe_num(bd_row["R02_steward_failures"]))),
    ("R03 security classification", int(safe_num(bd_row["R03_security_failures"]))),
    ("R04 schema name",             int(safe_num(bd_row["R04_schema_failures"]))),
    ("R05 row count",               int(safe_num(bd_row["R05_row_count_failures"]))),
    ("R06 PII flag",                int(safe_num(bd_row["R06_pii_failures"]))),
    ("R07 critical data flag",       int(safe_num(bd_row["R07_cde_failures"]))),
    ("R08 business term name",       int(safe_num(bd_row["R08_term_failures"]))),
    ("R09 PII security alignment",   int(safe_num(bd_row["R09_pii_security_failures"]))),
    ("R10 naming convention",        int(safe_num(bd_row["R10_naming_failures"]))),
    ("R11 table owner",              int(safe_num(bd_row["R11_table_owner_failures"]))),
    ("R12 tag value",                int(safe_num(bd_row["R12_tag_failures"]))),
]

worst_rule = max(rule_failures, key=lambda x: x[1])

print("=" * 70)
print("RULE BREAKDOWN — FAILURE COUNT PER RULE")
print("=" * 70)

for name, failures in rule_failures:
    pct = failures / total_full * 100 if total_full > 0 else 0
    print(f"{status_label(100 - pct):<14} | {name:<35} | {failures:>5} failures | {pct:>6.2f}%")

print("=" * 70)
print(f"Rule causing most failures : {worst_rule[0]} ({worst_rule[1]} failures)")
print("=" * 70)


# Table level metrics
table_metrics = df_full.groupBy("table_name").agg(
    count("*").alias("total_columns"),
    spark_sum(when(col("certification_status") == "PASS", 1).otherwise(0)).alias("passed_columns"),
    avg(when(col("certification_status") == "PASS", lit(100)).otherwise(lit(0))).alias("table_compliance_pct")
).orderBy("table_compliance_pct")

worst_table = table_metrics.orderBy("table_compliance_pct").first()
best_table = table_metrics.orderBy(col("table_compliance_pct").desc()).first()

print("=" * 70)
print("TABLE LEVEL METRICS")
print("=" * 70)

for row in table_metrics.collect():
    score = safe_num(row["table_compliance_pct"])
    table_name = row["table_name"] if row["table_name"] is not None else "Unknown table"

    print(
        f"{status_label(score):<14} | "
        f"{table_name:<35} | "
        f"{score:>6.2f}% | "
        f"{row['passed_columns']}/{row['total_columns']} columns passed"
    )

print("=" * 70)
print(f"Best table  : {best_table['table_name']} at {safe_num(best_table['table_compliance_pct']):.2f}%")
print(f"Worst table : {worst_table['table_name']} at {safe_num(worst_table['table_compliance_pct']):.2f}%")
print("=" * 70)


# Schema level metrics
schema_metrics = df_full.groupBy("schema_name").agg(
    count("*").alias("total_records"),
    spark_sum(when(col("certification_status") == "PASS", 1).otherwise(0)).alias("passed_records"),
    avg(when(col("certification_status") == "PASS", lit(100)).otherwise(lit(0))).alias("schema_compliance_pct")
).orderBy("schema_compliance_pct")

print("=" * 70)
print("SCHEMA LEVEL METRICS")
print("=" * 70)

for row in schema_metrics.collect():
    score = safe_num(row["schema_compliance_pct"])
    schema_name = row["schema_name"] if row["schema_name"] is not None else "Unknown schema"

    print(
        f"{status_label(score):<14} | "
        f"{schema_name:<25} | "
        f"{score:>6.2f}% | "
        f"{row['passed_records']}/{row['total_records']} records passed"
    )

print("=" * 70)


# Database level metrics
db_metrics = df_full.groupBy("database_name").agg(
    count("*").alias("total_records"),
    spark_sum(when(col("certification_status") == "PASS", 1).otherwise(0)).alias("passed_records"),
    avg(when(col("certification_status") == "PASS", lit(100)).otherwise(lit(0))).alias("database_compliance_pct")
).orderBy("database_compliance_pct")

print("=" * 70)
print("DATABASE LEVEL METRICS")
print("=" * 70)

for row in db_metrics.collect():
    score = safe_num(row["database_compliance_pct"])
    database_name = row["database_name"] if row["database_name"] is not None else "Unknown database"

    print(
        f"{status_label(score):<14} | "
        f"{database_name:<30} | "
        f"{score:>6.2f}% | "
        f"{row['passed_records']}/{row['total_records']} records passed"
    )

print("=" * 70)


# Overall catalog score
overall_score = safe_num(overall_rate)

print("=" * 70)
print("OVERALL CATALOG COMPLIANCE SUMMARY")
print("=" * 70)
print(f"Overall compliance score : {overall_score:.2f}%")
print(f"Total certified PASS     : {pass_full}")
print(f"Total FAIL               : {fail_full}")
print(f"Total records            : {total_full}")
print(f"Biggest governance gap   : {worst_field[0]} ({worst_field[1]:.2f}% complete)")
print(f"Most rule failures       : {worst_rule[0]} ({worst_rule[1]} records)")

if worst_table is not None:
    print(
        f"Worst performing table   : "
        f"{worst_table['table_name']} "
        f"({safe_num(worst_table['table_compliance_pct']):.2f}%)"
    )

print("=" * 70)

# Step 8: Write Certified Output to Gold 

Write two tables to the Gold layer — gold_certified_metadata and gold_quality_metrics — both with snapshot timestamps for the Metadata HUB history tracking.


In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

def safe_float(value, default=0.0):
    return float(value) if value is not None else default

def safe_int(value, default=0):
    return int(value) if value is not None else default


# Validate expected governance output columns
expected_output_cols = [
    "certification_status",
    "error_detail",
    "remediation_hint",
    "completeness_score",
    "ingestion_status"
]

missing_output = [c for c in expected_output_cols if c not in df_full.columns]

assert len(missing_output) == 0, \
    f"Missing output columns: {missing_output}"

print("Governance output schema validation PASSED")


# Add certification timestamp
df_final = df_full.withColumn(
    "certified_at",
    current_timestamp()
)


# Final Gold output tables
certified_table_name = "metadata_governance.gold.gold_certified_metadata_final"
metrics_table_name = "metadata_governance.gold.gold_quality_metrics_final"


# Write certified metadata results
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(certified_table_name)

certified_count = spark.table(certified_table_name).count()

assert certified_count == total_full, \
    f"Write count mismatch! Expected {total_full}, got {certified_count}"

print(f"Certified metadata table written successfully — {certified_count} rows")


# Build metrics dataset
metrics_rows = []


# Catalog level metrics
metrics_rows.append(Row(
    level="catalog",
    name="metadata_governance",
    total_records=safe_int(total_full),
    passed=safe_int(pass_full),
    failed=safe_int(fail_full),
    compliance_score=safe_float(overall_score)
))


# Schema level metrics
for row in schema_metrics.collect():

    score_col = (
        "schema_compliance_pct"
        if "schema_compliance_pct" in row.asDict()
        else "schema_compliance_%"
    )

    metrics_rows.append(Row(
        level="schema",
        name=str(row["schema_name"])
            if row["schema_name"] is not None
            else "Unknown schema",

        total_records=safe_int(row["total_records"]),
        passed=safe_int(row["passed_records"]),
        failed=safe_int(row["total_records"]) -
               safe_int(row["passed_records"]),

        compliance_score=safe_float(row[score_col])
    ))


# Table level metrics
for row in table_metrics.collect():

    score_col = (
        "table_compliance_pct"
        if "table_compliance_pct" in row.asDict()
        else "table_compliance_%"
    )

    metrics_rows.append(Row(
        level="table",
        name=str(row["table_name"])
            if row["table_name"] is not None
            else "Unknown table",

        total_records=safe_int(row["total_columns"]),
        passed=safe_int(row["passed_columns"]),
        failed=safe_int(row["total_columns"]) -
               safe_int(row["passed_columns"]),

        compliance_score=safe_float(row[score_col])
    ))


# Database level metrics
for row in db_metrics.collect():

    score_col = (
        "database_compliance_pct"
        if "database_compliance_pct" in row.asDict()
        else "database_compliance_%"
    )

    metrics_rows.append(Row(
        level="database",
        name=str(row["database_name"])
            if row["database_name"] is not None
            else "Unknown database",

        total_records=safe_int(row["total_records"]),
        passed=safe_int(row["passed_records"]),
        failed=safe_int(row["total_records"]) -
               safe_int(row["passed_records"]),

        compliance_score=safe_float(row[score_col])
    ))


# Rule level metrics
for rule_name, failures in rule_failures:

    failures = safe_int(failures)

    compliance = (
        (total_full - failures) / total_full * 100
        if total_full > 0 else 0
    )

    metrics_rows.append(Row(
        level="rule",
        name=str(rule_name),

        total_records=safe_int(total_full),
        passed=safe_int(total_full - failures),
        failed=safe_int(failures),

        compliance_score=safe_float(compliance)
    ))


# Field level metrics
for field_name, field_score in fields:

    field_score = safe_float(field_score)

    field_fail = (
        int((1 - field_score / 100) * total_full)
        if total_full > 0 else 0
    )

    metrics_rows.append(Row(
        level="field",
        name=str(field_name),

        total_records=safe_int(total_full),
        passed=safe_int(total_full - field_fail),
        failed=safe_int(field_fail),

        compliance_score=safe_float(field_score)
    ))


# Create metrics dataframe
df_metrics = spark.createDataFrame(metrics_rows) \
    .withColumn(
        "calculated_at",
        current_timestamp()
    )


# Write governance quality metrics
df_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(metrics_table_name)

metrics_count = spark.table(metrics_table_name).count()

print(f"Quality metrics table written successfully — {metrics_count} rows")


print("=" * 70)
print("STEP 8 — GOLD LAYER PROCESS COMPLETED")
print("=" * 70)
print(f"Gold base table          : metadata_governance.gold.gold_metadata")
print(f"Certified metadata table : {certified_table_name}")
print(f"Rows written             : {certified_count}")
print("-" * 70)
print(f"Quality metrics table    : {metrics_table_name}")
print(f"Metrics rows written     : {metrics_count}")
print("=" * 70)
print("Gold governance layer is now ready for:")
print("• Power BI dashboards")
print("• Databricks Genie AI")
print("• Governance reporting")
print("• Compliance analytics")
print("=" * 70)

# Step 9: Final Verification

Confirm all Gold tables were written correctly and display sample data for the demo.

In [0]:
from pyspark.sql.functions import col

gold_base    = spark.table("metadata_governance.gold.gold_metadata")
gold_cert    = spark.table("metadata_governance.gold.gold_certified_metadata_final")
gold_metrics = spark.table("metadata_governance.gold.gold_quality_metrics_final")

# Recalculate needed counts inside this cell
gold_count = gold_base.count()
certified_count = gold_cert.count()
metrics_count = gold_metrics.count()

pass_full = gold_cert.filter(col("certification_status") == "PASS").count()
fail_full = gold_cert.filter(col("certification_status") == "FAIL").count()

total_full = pass_full + fail_full

quarantine_count = gold_cert.filter(col("ingestion_status") == "FAILED_SILVER").count()

overall_score = (pass_full / total_full * 100) if total_full > 0 else 0


print("=" * 70)
print("GOLD LAYER VALIDATION SUMMARY")
print("=" * 70)
print(f"Gold base metadata table        : {gold_count} rows")
print(f"Certified governance records    : {certified_count} rows")
print(f"Quality metrics records         : {metrics_count} rows")
print("=" * 70)

print("\nCertified metadata schema:")
gold_cert.printSchema()

print("\nSample PASS records — metadata successfully governed and certified")
display(
    gold_cert.filter(
        col("certification_status") == "PASS"
    ).select(
        "table_name",
        "column_name",
        "certification_status",
        "completeness_score",
        "security_classification",
        "data_steward",
        "ingestion_status",
        "certified_at"
    ).limit(5)
)

print("\nSample FAIL records — governance issues requiring attention")
display(
    gold_cert.filter(
        col("certification_status") == "FAIL"
    ).select(
        "table_name",
        "column_name",
        "certification_status",
        "completeness_score",
        "error_detail",
        "remediation_hint",
        "ingestion_status",
        "certified_at"
    ).limit(5)
)

print("\nGovernance metrics by rule")
display(
    gold_metrics.filter(
        col("level") == "rule"
    ).orderBy("compliance_score")
)

print("\nGovernance metrics by table")
display(
    gold_metrics.filter(
        col("level") == "table"
    ).orderBy("compliance_score")
)

print("\nCatalog governance overview")
display(
    gold_metrics.filter(
        col("level") == "catalog"
    )
)

print("=" * 70)
print("FINAL GOVERNANCE PIPELINE SUMMARY")
print("=" * 70)
print(f"Original Bronze records processed : {total_full}")
print(f"Records passed Silver ingestion   : {gold_count}")
print(f"Records sent to quarantine        : {quarantine_count}")
print("-" * 70)
print(f"Governance PASS records           : {pass_full}")
print(f"Governance FAIL records           : {fail_full}")
print(f"Overall compliance score          : {overall_score:.2f}%")
print("-" * 70)
print(f"Governance rules applied          : 12")
print(f"Gold layer tables created         : 3")
print(f"Quality metric records generated  : {metrics_count}")
print("=" * 70)
print("Governance pipeline completed successfully")
print("=" * 70)